# Exercise Sheet No. 7

---

> Machine Learning for Natural Sciences, Prof. Pascal Friederich
>
> Deadline: see the course announcement
>
> Tutor: yumeng.zhang@kit.edu
>
> Please ask questions in the forum/discussion board. For grading issues, contact the tutor.

---

**Topic**: This assignment focuses on convolutional neural networks for breast tumor image classification.

Please add here your group members' names and student IDs. 

Names: 

IDs:

# Preliminaries
In this assignment, we are going to use PyTorch instead of TensorFlow to build our neural network.

In [ ]:
##### DO NOT CHANGE #####
import os
os.environ["CUDA_VISIBLE_DEVICES"]=""

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import PIL
from PIL import Image
from matplotlib.pyplot import MultipleLocator
import matplotlib.image as mpimg
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using {device} device')
print(PIL.__version__)

##### DO NOT CHANGE #####

For the purpose of autograding, please submit your notebook with `do_training = False`.

While you are working on the exercise, you may temporarily set `do_training = True` to run the training cells and inspect the learning curves. With `do_training = False`, the training cells are skipped so the submitted notebook remains fast. Before submission, change it back to `False` and run the notebook once from top to bottom.

In [ ]:
# Keep this False for submission. You may temporarily set it to True while exploring.
do_training = False
# YOUR CODE HERE
raise NotImplementedError()

In the last assignments, we have implemented and trained fully connected neural networks with applications in chemistry and materials science. This time, we will learn to implement and train a convolutional neural network (CNN) for breast tumor image classification, and compare it with a fully connected neural network to show the power of convolutional filters. In addition to that, we will see how a pretrained network can be used to fulfill the same task with better performance through transfer learning.

# Dataset
In this assignment, we will work with breast tumor ultrasound images from healthy patients and cancer patients.

Breast cancer is the most common cancer in women globally and can often be treated successfully when detected early and still localized. However, advanced breast cancer that has spread to distant organs is much harder to treat. You may read more about breast cancer in this [paper](https://www.nature.com/articles/s41572-019-0111-2).

The original dataset is available on [Kaggle](https://www.kaggle.com/datasets/aryashah2k/breast-ultrasound-images-dataset). To avoid Kaggle login or timeout problems during the exercise, this notebook downloads a mirrored copy from KIT/bwSync and verifies it with a file-size and checksum check. We use only the `normal` and `malignant` images, and we exclude segmentation mask images. Now, let's get started by visualizing part of the dataset.

In [ ]:
##### DO NOT CHANGE #####
import hashlib
import shutil
import time
import zipfile
from collections import Counter
from pathlib import Path

import requests

DATA_DIR = Path("dataset_breast")
DATASET_ZIP = Path("breast-ultrasound-images-dataset.zip")
BWSYNC_DATASET_URL = "https://bwsyncandshare.kit.edu/s/6eDdrMaCkqgF3g3/download"
KAGGLE_DATASET_URL = "https://www.kaggle.com/api/v1/datasets/download/aryashah2k/breast-ultrasound-images-dataset"
DATASET_URLS = [
    ("KIT/bwSync mirror", BWSYNC_DATASET_URL),
    ("Kaggle", KAGGLE_DATASET_URL),
]
EXPECTED_DATASET_SIZE = 204421470
EXPECTED_DATASET_SHA256 = "7fffc86a517934da55021f66641d81383058d26a30adb0ffab3780ca7e52d57d"
EXPECTED_LABEL_COUNTS = {"malignant": 210, "normal": 133}
EXPECTED_NUM_IMAGES = sum(EXPECTED_LABEL_COUNTS.values())


def print_progress(done, total, prefix="", width=30):
    if total:
        fraction = min(done / total, 1.0)
        filled = int(width * fraction)
        bar = "#" * filled + "-" * (width - filled)
        print(f"\r{prefix} [{bar}] {100 * fraction:5.1f}%", end="", flush=True)
    else:
        print(f"\r{prefix} {done / (1024 * 1024):.1f} MB", end="", flush=True)


def file_sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def dataset_zip_is_valid(path):
    if not path.exists():
        return False
    if path.stat().st_size != EXPECTED_DATASET_SIZE:
        print(f"Found {path.name}, but the file size is not correct.")
        return False
    if file_sha256(path) != EXPECTED_DATASET_SHA256:
        print(f"Found {path.name}, but the checksum is not correct.")
        return False
    return True


def dataset_label_counts(path):
    files = list(path.glob("*.png")) if path.exists() else []
    return Counter(file.name.split(" ")[0] for file in files)


def extracted_dataset_is_valid(path):
    if not path.exists():
        return False
    files = list(path.glob("*.png"))
    label_counts = Counter(file.name.split(" ")[0] for file in files)
    return len(files) == EXPECTED_NUM_IMAGES and dict(label_counts) == EXPECTED_LABEL_COUNTS


def download_dataset(urls, path, attempts_per_source=3):
    tmp_path = path.with_suffix(path.suffix + ".tmp")
    errors = []

    for source_name, url in urls:
        print(f"Downloading the breast ultrasound dataset from {source_name}...")
        for attempt in range(1, attempts_per_source + 1):
            try:
                if tmp_path.exists():
                    tmp_path.unlink()
                print(f"Download attempt {attempt}/{attempts_per_source}")
                response = requests.get(url, stream=True, timeout=(30, 300))
                response.raise_for_status()

                total_size = int(response.headers.get("content-length", EXPECTED_DATASET_SIZE))
                downloaded = 0
                with tmp_path.open("wb") as file:
                    for chunk in response.iter_content(chunk_size=1024 * 1024):
                        if not chunk:
                            continue
                        file.write(chunk)
                        downloaded += len(chunk)
                        print_progress(downloaded, total_size, prefix="Download")
                print()

                tmp_path.replace(path)
                if dataset_zip_is_valid(path):
                    return

                path.unlink(missing_ok=True)
                raise RuntimeError("downloaded file failed the size/checksum check")
            except (requests.RequestException, RuntimeError) as error:
                errors.append(f"{source_name}, attempt {attempt}: {error}")
                print(f"Download attempt {attempt} from {source_name} failed: {error}")
                if attempt < attempts_per_source:
                    time.sleep(5)

    raise RuntimeError(
        "Could not download the dataset from any source. Please check your internet "
        "connection, or ask the tutor for the dataset archive. Details:\n"
        + "\n".join(errors)
    )


def extract_breast_subset(dataset_zip, output_dir):
    print("Extracting normal and malignant non-mask images...")
    if output_dir.exists():
        shutil.rmtree(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(dataset_zip) as archive:
        members = archive.namelist()
        selected_members = []
        for member in members:
            member_lower = member.lower()
            if not member_lower.endswith(".png") or "_mask" in member_lower:
                continue
            label = Path(member).name.split(" ")[0]
            if label in EXPECTED_LABEL_COUNTS:
                selected_members.append(member)

        for index, member in enumerate(selected_members, start=1):
            filename = Path(member).name
            with archive.open(member) as source, (output_dir / filename).open("wb") as target:
                shutil.copyfileobj(source, target)
            print_progress(index, len(selected_members), prefix="Extract")
    print()


print("Preparing breast ultrasound dataset...")
if DATA_DIR.exists():
    print(f"Found existing {DATA_DIR}. Skipping download.")
    if not extracted_dataset_is_valid(DATA_DIR):
        label_counts = dataset_label_counts(DATA_DIR)
        raise RuntimeError(
            f"{DATA_DIR} exists, but it is incomplete or has wrong label counts: "
            f"{dict(label_counts)}. Please delete this folder and rerun the cell."
        )
else:
    print(f"No extracted dataset found at {DATA_DIR}.")
    if DATASET_ZIP.exists():
        print(f"Found existing archive {DATASET_ZIP}. Skipping download.")
        if not dataset_zip_is_valid(DATASET_ZIP):
            raise RuntimeError(
                f"{DATASET_ZIP} exists, but its size/checksum is wrong. "
                "Please delete this file and rerun the cell to download it again."
            )
    else:
        download_dataset(DATASET_URLS, DATASET_ZIP)

    print("Checking dataset archive size and checksum...")
    if not dataset_zip_is_valid(DATASET_ZIP):
        raise RuntimeError("The dataset archive does not match the expected size/checksum.")
    print("Dataset archive verified.")

    extract_breast_subset(DATASET_ZIP, DATA_DIR)

if not extracted_dataset_is_valid(DATA_DIR):
    raise RuntimeError("The extracted dataset is incomplete or in the wrong location.")

label_counts = dataset_label_counts(DATA_DIR)
print(f"Dataset ready: {EXPECTED_NUM_IMAGES} images in {DATA_DIR}.")
print("Label counts:", dict(label_counts))

img_dir = os.path.realpath(DATA_DIR)  # path of image directory
images = os.listdir(img_dir)

##### DO NOT CHANGE #####

In [ ]:
##### DO NOT CHANGE #####
images = sorted(images)

# plot 10 ultrasound images
fig, axes = plt.subplots(2, 5, figsize=(40, 20), gridspec_kw=dict(hspace=0.1, wspace=0.3))
for i, ax in enumerate(axes.ravel()):
    ax.imshow(Image.open(os.path.join(img_dir, images[i])))
    ax.set_title(images[i])

##### DO NOT CHANGE #####

## Principal Component Analysis (PCA)
From the plot above we can see that images in the dataset vary by several features, such as brightness, orientation, and of course whether they have a tumor or not. For such a complicated dataset, is it possible to classify it with a simple classifier like k-nearest neighbors? Principal Component Analysis can give us a hint on this problem.

PCA is an unsupervised technique to project the data to lower dimensional space through linear (linear PCA) or non-linear (kernel PCA) combinations of the data's original features. The "Principal Component" stands for combination results of features that account for most of a dataset's variance. PCA can also be used to visualize high-dimensional data, filter out noise, or as feature representation. Here, we will take this advantage to visualize the most significant features in our dataset.

In [ ]:
##### DO NOT CHANGE #####
# prepare data in numpy array for PCA()
data = [Image.open(os.path.join(img_dir, i)).convert('L').resize((256, 256)) for i in images]
data = np.array([np.array(d).reshape(-1) for d in data])

##### DO NOT CHANGE #####

Please use the `PCA()` to perform principal component analysis on `data`. Use the parameter `n_components` to set the amount of variance that needs to be explained to be 0.8.

In [ ]:
pca = None

# YOUR CODE HERE
raise NotImplementedError()

In [ ]:
##### DO NOT CHANGE #####
# ID: pca - possible points: 1

assert isinstance(pca, PCA), "pca should be an instance of the sklearn PCA"

##### DO NOT CHANGE #####

Now, let's visualize the images associated with the first several principal components, which may give us some insight about what make these images vary from each other.

In [ ]:
##### DO NOT CHANGE #####
# visualize the first 10 principal components
fig, axes = plt.subplots(2, 5, figsize=(40, 20), gridspec_kw=dict(hspace=0.1, wspace=0.1))
for i, ax in enumerate(axes.ravel()):
    ax.imshow(pca.components_[i].reshape(256, 256), cmap='gray')

##### DO NOT CHANGE #####

As shown above, some principal components seem to be associated with brightness. Since none of them shows a clear relationship to the shape, size, or location of the tumor, it is difficult to classify cancer/not-cancer images with a simple algorithm that focuses only on the most obvious global features. As a result, a more flexible model such as a neural network is suitable for this task.

## Data processing
To begin with, we need to obtain and organize all image metadata with `pandas`. Please use `pd.Series` for `img_names` and `img_labels`, together with `pd.concat()` and `pd.DataFrame` to construct `tumor_df`.

In [ ]:
# implement tumor_df as pd.DataFrame for train/test split and further data process
tumor_df = None
img_names, img_labels = zip(*[(i, 0 if 'normal' in i else 1) for i in images])

# YOUR CODE HERE
raise NotImplementedError()

In [ ]:
##### DO NOT CHANGE #####
# ID: dataframe - possible points: 1

assert len(tumor_df) == 343, 'Please check the set up of tumor_df'
assert tumor_df["label"].value_counts()[1] == 210
assert tumor_df["label"].value_counts()[0] == 133

##### DO NOT CHANGE #####

Now, for the training/test set generation, please use `train_test_split()` to split the `tumor_df` into `train_set` and `test_set`. This time, we will use 80% of the data for training and the rest for testing.

Please fix `random_state` to `0` and use `stratify=tumor_df["label"]`, so the train and test sets keep a similar normal/malignant class ratio.

In [ ]:
# training set/test set split
train_set, test_set = None, None

# YOUR CODE HERE
raise NotImplementedError()

In [ ]:
##### DO NOT CHANGE #####
# ID: data-split - possible points: 1

# train-test split 1 point
assert train_set.shape == (274, 2)
assert test_set.shape == (69, 2)
assert train_set["label"].value_counts().sort_index().to_dict() == {0: 106, 1: 168}
assert test_set["label"].value_counts().sort_index().to_dict() == {0: 27, 1: 42}


##### DO NOT CHANGE #####

PyTorch offers convenient APIs for handling data: `Dataset` and `DataLoader`, which make data loading and preprocessing more readable and modular. Take the following implementation as an example. Note that this implementation is only for demonstration. Since our data fits into memory, a more performant implementation would use a `TensorDataset` or cache images. Repeatedly loading samples from disk is inefficient and slows down training.

In [ ]:
##### DO NOT CHANGE #####
class TumorImageDataset(Dataset):
    """load, transform and return image and label"""

    def __init__(self, annotations_df, img_dir, transform=None):
        self.img_labels = annotations_df
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.img_labels)

    def __getitem__(self, idx):
        # get image path according to idx
        img_path = os.path.join(self.img_dir, self.img_labels.iloc[idx, 0])
        # convert all image to RGB format
        image = Image.open(img_path).convert('RGB')
        label = self.img_labels.iloc[idx, 1]
        # apply image transform
        if self.transform:
            image = self.transform(image)
        return [image, label]

##### DO NOT CHANGE #####

Please finish the `image_transform` module. The image should be resized to $256\times 256$ then cropped at center to size $244 \times 244$. You may use `transforms.Resize()` and `transforms.CenterCrop()` for this task.

In [ ]:
# user defined image transform process
# here we resize and cut the center of each image to obtain a dataset with uniform size
image_transform = transforms.Compose([
    # YOUR CODE HERE
    raise NotImplementedError()
    transforms.ToTensor()
])

In [ ]:
##### DO NOT CHANGE #####
# implement Dataset and DataLoader for training
train_data = TumorImageDataset(train_set, img_dir, image_transform)
train_dataloader = DataLoader(train_data, batch_size=32, shuffle=False)

test_data = TumorImageDataset(test_set, img_dir, image_transform)
test_dataloader = DataLoader(test_data, batch_size=32, shuffle=False)

image_datasets = {'train': train_data, 'test': test_data}
image_dataloaders = {'train': train_dataloader, 'test': test_dataloader}

##### DO NOT CHANGE #####

In [ ]:
##### DO NOT CHANGE #####
# ID: data-transform - possible points: 1

feat, labels = next(iter(train_dataloader))
assert feat.shape[2] == 244 and feat.shape[3] == 244, "Wrong size, please check image_transform"

##### DO NOT CHANGE #####

Here is a plot for part of the dataset with `train_dataloader`.

In [ ]:
##### DO NOT CHANGE #####
# plot a batch (32) of ultrasound images in the training set
train_features, train_labels = next(iter(train_dataloader))  # DataLoader is iterable
fig, axes = plt.subplots(4, 8, figsize=(40, 20), gridspec_kw=dict(hspace=0.1, wspace=0.3))
for i, ax in enumerate(axes.flat):
    ax.imshow(train_features[i].numpy().transpose((1, 2, 0)))
    ax.set_title('Cancer' if int(train_labels[i] == 1) else 'Not Cancer')

##### DO NOT CHANGE #####

# Build the neural network
## motivation of CNN: compare with fully connected layers
As discussed in the class, an advantage of convolutional neural network (CNN) is weight-sharing of filter parameters. Let's demonstrate this effect by looking at a densely connected NN.

PyTorch builds neural networks by subclassing `nn.Module`. For each `nn.Module`, the `forward()` method defines the operations applied to input data.

We implement the `MultiLayerPerceptron` with three fully connected layers. Please finish the `forward()` method using `self.fc1`, `self.fc2`, and `self.fc3`, which have already been defined. For the first two layers, use `F.relu()` as the activation function. For the output layer, use `torch.sigmoid()` to generate an output between 0 and 1.

In [ ]:
class MultiLayerPerceptron(nn.Module):
    """A three layer fully connected neural network"""

    def __init__(self):
        super(MultiLayerPerceptron, self).__init__()
        self.fc1 = nn.Linear(in_features=244 * 244 * 3, out_features=256)
        self.fc2 = nn.Linear(in_features=256, out_features=64)
        self.fc3 = nn.Linear(in_features=64, out_features=1)

    def forward(self, x):
        """Operations on x"""
        x = torch.flatten(x, 1)
        # YOUR CODE HERE
        raise NotImplementedError()
        return x

In [ ]:
##### DO NOT CHANGE #####
# ID: mlp-train - possible points: 3

mlp_model = MultiLayerPerceptron().to(device)

assert len(mlp_model.state_dict().keys()) == 6, 'please check if there is any fc layer mising in forward()'


##### DO NOT CHANGE #####

PyTorch does not provide a built-in method to display model information in exactly the same way as the `summary()` method in TensorFlow. Luckily, a small helper function is enough for our purposes. Here is an example:

In [ ]:
##### DO NOT CHANGE #####
def summary(model):
    """Print out model architecture information"""
    parameter_count = 0
    model_info = model.state_dict()
    for name, module in model.named_children():
        # loop each module in the model to record number of parameters
        try:
            n_weight = model_info[name + '.weight'].flatten().shape[0]
            n_bias = model_info[name + '.bias'].flatten().shape[0]
        except:
            n_weight = 0
            n_bias = 0
        print(f'{name} layer (No. of weight: {n_weight:n}, No. of bias: {n_bias:n})')
        parameter_count += (n_weight + n_bias)
    print(f'Total parameters: {parameter_count:n}')

##### DO NOT CHANGE #####

In [ ]:
##### DO NOT CHANGE #####
summary(mlp_model)

##### DO NOT CHANGE #####

## Training
As seen from the `summary()` result, dense networks for images can become very large, with more than $10^7$ parameters in this case. This can lead to long training times.

If you temporarily set `do_training = True`, the next cell trains this network for 1 epoch using the `train_model()` function defined below. With `do_training = False`, this training call is skipped for submission.

In [ ]:
##### DO NOT CHANGE #####
def train_model(model, loss_func, optimizer, epochs, image_datasets, image_dataloaders, do_training=True):
    """Return the trained model and train/test accuracy/loss"""
    if not do_training:
        return None, None
    history = {'train_loss': [], 'train_acc': [], 'test_loss': [], 'test_acc': []}
    for e in range(1, epochs + 1):
        print('Epoch {}/{}'.format(e, epochs))
        for phase in ['train', 'test']:
            if phase == 'train':
                model.train()  # set model to training mode for training phase
            else:
                model.eval()  # set model to evaluation mode for test phase

            running_loss = 0.0  # record the training/test loss for each epoch
            running_corrects = 0  # record the number of correct predicts by the model for each epoch

            for features, labels in image_dataloaders[phase]:
                # send data to gpu if possible
                features = features.to(device)
                labels = labels.to(device)

                # reset the parameter gradients after each batch to avoid double-counting
                optimizer.zero_grad()

                # forward pass
                # set parameters to be trainable only at training phase
                with torch.set_grad_enabled(phase == 'train'):
                    outcomes = model(features)
                    pred_labels = outcomes.round()  # round up forward outcomes to get predicted labels
                    labels = labels.unsqueeze(1).type(torch.float)
                    loss = loss_func(outcomes, labels)  # calculate loss

                    # backpropagation only for training phase
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                # record loss and correct predicts of each bach
                running_loss += loss.item() * features.size(0)
                running_corrects += torch.sum(pred_labels == labels.data)

            # record loss and correct predicts of each epoch and stored in history
            epoch_loss = running_loss / len(image_datasets[phase])
            epoch_acc = running_corrects.double() / len(image_datasets[phase])

            print('{} Loss: {:.4f} Acc: {:.4f}'.format(phase, epoch_loss, epoch_acc))
            history[phase + '_loss'].append(epoch_loss)
            history[phase + '_acc'].append(epoch_acc)

    return model, history

##### DO NOT CHANGE #####

For binary classification, we will use Binary Cross Entropy `BCELoss()` as loss function (read more information in [documentation](https://pytorch.org/docs/stable/generated/torch.nn.BCELoss.html)). We use `optim.Adam()` for optimization. For a more detailed explanation, you may read the original paper from 2015 [here](https://arxiv.org/abs/1412.6980) .

In [ ]:
##### DO NOT CHANGE #####
mlp_model_trained, history = train_model(
    model=mlp_model,
    loss_func=nn.BCELoss(),
    optimizer=optim.Adam(mlp_model.parameters(), lr=0.001),
    epochs=1,
    image_datasets=image_datasets,
    image_dataloaders=image_dataloaders,
    do_training=do_training
)

##### DO NOT CHANGE #####

## Convolutional Neural Network 
Now, let's build our own CNN `TumorNet`. Please finish the convolutional part in `forward()`:
1. Use `self.conv` as the convolutional layer with the `F.relu()` activation function.
2. Add max pooling layer `self.pool`. 
3. Flatten the feature map and feed it into `self.fc1` with the `F.relu()` activation function.
4. Use `self.dropout` for regularization.
5. Use `self.fc2` as the output layer with `torch.sigmoid()` as activation function.

This small network is intentionally simple: one convolutional block extracts local image features, and the fully connected part maps those features to one binary output.

<img src="https://bwsyncandshare.kit.edu/s/L9Gt2zY8SH5w3n3/download" alt="TumorNet CNN architecture diagram">

In [ ]:
class TumorNet(nn.Module):
    """
    A CNN with:
        one convolutional layer
        one max pooling layer
        two fully connected layers
    """

    def __init__(self):
        super(TumorNet, self).__init__()
        self.conv = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=5, stride=2)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(in_features=60 * 60 * 16, out_features=64)
        self.dropout = nn.Dropout(p=0.3)
        self.fc2 = nn.Linear(in_features=64, out_features=1)

    def forward(self, x):
        """Operations on x"""
        # YOUR CODE HERE
        raise NotImplementedError()
        return x

In [ ]:
##### DO NOT CHANGE #####
# ID: tumornet-train - possible points: 3

cnn_model = TumorNet().to(device)
summary(cnn_model)

assert 'conv.weight' in cnn_model.state_dict().keys()


##### DO NOT CHANGE #####

In [ ]:
##### DO NOT CHANGE #####
cnn_model_trained, cnn_history = train_model(
    model=cnn_model,
    loss_func=nn.BCELoss(),
    optimizer=optim.Adam(cnn_model.parameters(), lr=0.001),
    epochs=15,
    image_datasets=image_datasets,
    image_dataloaders=image_dataloaders,
    do_training=do_training
)

##### DO NOT CHANGE #####

A key advantage of convolutional neural networks is parameter sharing. Compute the total number of trainable and non-trainable parameters in the fully connected `mlp_model` and in the convolutional `cnn_model`, then compare them.

In [ ]:
mlp_num_parameters = 0
cnn_num_parameters = 0
# YOUR CODE HERE
raise NotImplementedError()

In [ ]:
##### DO NOT CHANGE #####
# ID: acc-test-1 - possible points: 2

assert np.asarray(mlp_num_parameters).shape == ()
assert np.asarray(cnn_num_parameters).shape == ()
assert mlp_num_parameters > 0
assert cnn_num_parameters > 0


##### DO NOT CHANGE #####

If you run training with `do_training = True`, we can plot the training curves to visualize loss and accuracy over epochs for both the training and test sets. The hyperparameters of this model are not optimized. Feel free to try hyperparameter optimization in another notebook, but keep this submitted notebook unchanged except for the required answer cells.

In [ ]:
##### DO NOT CHANGE #####
def plot_training_curve(history):
    """Plot the training curve"""
    train_loss = history['train_loss']
    test_loss = history['test_loss']
    train_acc = history['train_acc']
    test_acc = history['test_acc']

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 5))
    ax1.plot(list(range(1, len(train_loss) + 1)), train_loss, label='Training', color='c')
    ax1.plot(list(range(1, len(train_loss) + 1)), test_loss, label='Test', color='b')
    x_major_locator = MultipleLocator(1)
    ax1.set_xlim(1, len(train_loss))
    ax1.xaxis.set_major_locator(x_major_locator)
    ax1.set_xlabel('Epochs')
    ax1.set_ylabel('Binary Cross Entropy Loss')
    ax1.legend(loc='upper right', fontsize='x-large')
    ax1.set_title('Loss vs. Epochs')

    ax2.plot(np.arange(1, len(train_acc) + 1), train_acc, label='Training', color='c')
    ax2.plot(np.arange(1, len(train_acc) + 1), test_acc, label='Test', color='b')
    x_major_locator = MultipleLocator(1)
    ax2.set_xlim(1, len(train_acc))
    ax2.xaxis.set_major_locator(x_major_locator)
    ax2.set_xlabel('Epochs')
    ax2.set_ylabel('Accuracy')
    ax2.legend(loc='lower right', fontsize='x-large')
    ax2.set_title('Accuracy vs. Epochs')

    plt.show()
    plt.close()

##### DO NOT CHANGE #####

In [ ]:
##### DO NOT CHANGE #####
try:
    plot_training_curve(cnn_history)
except:
    pass

##### DO NOT CHANGE #####

## Transfer learning
In practice, people often start with pretrained models instead of training an entire convolutional network from scratch. Transfer learning reuses representations learned from a large dataset and adapts them to a smaller target dataset. This can improve performance and speed up convergence, especially when the target dataset is small, as in this assignment.

There are two major scenarios where transfer learning is used:
1. Finetuning the CNN: instead of random initialization, the network is initialized with pretrained weights and then trained on the target dataset. In this scenario, parameters in both convolutional and fully connected layers are trainable.
2. Using the CNN as a feature extractor: all layers except the last fully connected layer are frozen. The last fully connected layer is replaced to match the target task, and only this new layer is trained.

For more information about transfer learning, take a look at this [note](https://cs231n.github.io/transfer-learning/).

In this assignment, we are going to take the second choice and use ResNet18 as our pretrained feature extractor. 
ResNet, or residual network, is a neural network architecture that applies identity mapping as a "shortcut connection" (see figure below). This architecture helps with the vanishing-gradient problem in deep neural networks. For more information, please read the original [paper](https://arxiv.org/abs/1512.03385).

<img src="https://bwsyncandshare.kit.edu/s/nEwic4XNBNCGGqZ/download" alt="ResNet architecture diagram">

To use ResNet18 as a feature extractor, we first build a ResNet18 model and freeze its existing parameters. When `do_training = True`, the notebook tries to load ImageNet-pretrained weights. When `do_training = False`, it uses the same ResNet18 architecture without downloading weights, so the submitted notebook remains fast and reproducible.

Please finish the remaining steps as described here:
1. Replace the last fully connected layer with the desired module for the tumor classification task. You may use PyTorch `nn.Sequential` to concatenate `nn.Linear(in_features, out_features)` and `nn.Sigmoid()`. You may use `resnet_model.fc.in_features` to obtain `in_features`. The `out_features` should be 1 because we are working on a binary classification problem with one output node.
2. Assign the resulting `Sequential` module to the last layer of the model, `resnet_model.fc`.

Do not worry about setting the parameters of the last layer to be trainable, as newly constructed modules have `requires_grad=True` by default.

In [ ]:
def build_resnet18(use_pretrained=False):
    if use_pretrained:
        try:
            weights = torchvision.models.ResNet18_Weights.DEFAULT
            return torchvision.models.resnet18(weights=weights)
        except Exception as error:
            print(
                "Could not load pretrained ResNet18 weights. "
                f"Using randomly initialized weights instead. ({error})"
            )

    try:
        return torchvision.models.resnet18(weights=None)
    except TypeError:
        # Compatibility with older torchvision versions.
        return torchvision.models.resnet18(pretrained=False)


# Load pretrained weights only during exploration/training. In submission mode,
# avoid any network download and build the same architecture with random weights.
resnet_model = build_resnet18(use_pretrained=do_training).to(device)
for p in resnet_model.parameters():
    # freeze all parameters
    p.requires_grad = False

# YOUR CODE HERE
raise NotImplementedError()

If you temporarily set `do_training = True`, the next cell trains the ResNet classification head and lets you compare it with the CNN model implemented earlier. With `do_training = False`, this training call is skipped for submission.

In [ ]:
##### DO NOT CHANGE #####
resnet_trained, resnet_history = train_model(
    model=resnet_model,
    loss_func=nn.BCELoss(),
    optimizer=optim.Adam(resnet_model.parameters(), lr=0.001),
    epochs=20,
    image_datasets=image_datasets,
    image_dataloaders=image_dataloaders,
    do_training=do_training
)

##### DO NOT CHANGE #####

Instead of submitting a manually typed ResNet test accuracy, verify that the ResNet feature extractor was set up correctly. The original ResNet parameters should be frozen, while the newly added classification head should remain trainable.

In [ ]:
resnet_feature_extractor_ready = False
# YOUR CODE HERE
raise NotImplementedError()

In [ ]:
##### DO NOT CHANGE #####
# ID: acc-test-2 - possible points: 4

assert resnet_feature_extractor_ready == True


##### DO NOT CHANGE #####

In [ ]:
##### DO NOT CHANGE #####
try:
    plot_training_curve(resnet_history)
except:
    pass

##### DO NOT CHANGE #####

## Application: classify the held-out test set

A trained CNN is useful only if we can apply it to new images. In this section, you will turn the TumorNet CNN from above into a small image classifier and apply it to every image in the held-out `test_set`.

The labels are encoded as `0 = normal` and `1 = malignant`. If you are exploring with `do_training = True`, you can inspect the model predictions and compare them with the true test-set labels. For submission with `do_training = False`, grading checks that your inference pipeline returns valid probabilities and thresholded labels. It does not grade prediction accuracy.

In [ ]:
##### DO NOT CHANGE #####
application_test_set = test_set.copy().reset_index(drop=True)
application_test_image_paths = [
    os.path.join(img_dir, image_name)
    for image_name in application_test_set["image_name"]
]
application_model = cnn_model_trained if cnn_model_trained is not None else cnn_model

fig, axes = plt.subplots(1, 5, figsize=(20, 4), gridspec_kw=dict(wspace=0.3))
for ax, (_, row) in zip(np.atleast_1d(axes), application_test_set.head(5).iterrows()):
    image_path = os.path.join(img_dir, row["image_name"])
    ax.imshow(Image.open(image_path).convert("RGB"))
    ax.set_title(f"label: {int(row['label'])}")
    ax.axis("off")
plt.show()

##### DO NOT CHANGE #####

Implement `predict_tumor_image()` so that it loads one image, applies the same `image_transform`, adds a batch dimension, runs the model in evaluation mode without gradients, and returns:

1. `probability`: the model output as a scalar probability between 0 and 1.
2. `predicted_label`: `0` or `1`, using a threshold of `0.5`.

In the next answer cell, apply `predict_tumor_image()` to every path in `application_test_image_paths`. Store the predicted probabilities in `application_probabilities` and the predicted labels in `application_predicted_labels`, preserving the test-set order. These can be Python lists or NumPy arrays. Do not copy the true labels here; the locked test cell will read them from `application_test_set`.

In [ ]:
def predict_tumor_image(model, image_path, transform, device):
    probability = None
    predicted_label = None

    # YOUR CODE HERE
    raise NotImplementedError()

    return probability, predicted_label

In [ ]:
application_probabilities = None
application_predicted_labels = None

# YOUR CODE HERE
raise NotImplementedError()

In [ ]:
##### DO NOT CHANGE #####
# ID: application-inference - possible points: 1

application_probabilities = np.asarray(application_probabilities, dtype=float).reshape(-1)
application_predicted_labels = np.asarray(application_predicted_labels, dtype=float).reshape(-1)
application_test_labels = application_test_set["label"].to_numpy(dtype=int)

print(f"Predicted {len(application_predicted_labels)} test-set labels.")
print(f"First five predicted labels: {application_predicted_labels[:5].astype(int).tolist()}")
print(f"First five true labels: {application_test_labels[:5].tolist()}")

assert application_probabilities.shape == (len(application_test_set),)
assert np.all((0.0 <= application_probabilities) & (application_probabilities <= 1.0))
assert application_predicted_labels.shape == (len(application_test_set),)
assert np.all(np.isclose(application_predicted_labels, np.round(application_predicted_labels)))
assert set(np.unique(application_predicted_labels)).issubset({0.0, 1.0})
assert application_test_labels.shape == (len(application_test_set),)
assert set(np.unique(application_test_labels)).issubset({0, 1})


##### DO NOT CHANGE #####

In [ ]:
##### DO NOT CHANGE #####
# ID: do-training-false - possible points: 1

# Please set do_training at the beginning of this assignment to False. Thank you!
assert do_training == False

##### DO NOT CHANGE #####

Hope you enjoy this assignment. Thank you very much!
**This is the end of the assignment**